# 10 — Block-Scaled Matrix Multiplication

**Ladder:** Scale-then-matmul (materialize) → Triton scale + tutorial-03 GEMM → (official) in-kernel FP8 / block scales fused into the $K$-loop.

**Study tip.** Master the scalar identity $(s_A s_B)AB$ first. Materializing full scaled $A,B$ teaches cost; real FP8 path fuses scales inside the $K$-loop (official tutorial).

Official: [Block-Scaled Matmul](https://triton-lang.org/main/getting-started/tutorials/10-block-scaled-matmul.html)

Primer: [`../03_matrix_multiplication/README.md`](../03_matrix_multiplication/README.md) · figures: [`docs/images/`](docs/images/)

![Whiteboard: scale-then-GEMM vs in-kernel block scales](docs/images/benchmark-block-scaled-matmul.png)

*Theory board — materialize scaled $A,B$ vs dequant with per-block scales inside the $K$-loop.*

---

## 0. Scalar-scaled product (teaching form)

For $s_A, s_B \in \mathbb{R}$ and $A \in \mathbb{R}^{M \times K}$, $B \in \mathbb{R}^{K \times N}$:

$$
C = (s_A A)\,(s_B B) = (s_A s_B)\, (A B).
$$

**Homogeneity.** Scaling inputs by constants factors out of the bilinear map (exact in $\mathbb{R}$; FP only reorders rounding).

**Block scales (FP8 production form).** If each tile $(p_m,t)$ of $A$ has scale $s^{A}_{p_m,t}$ and each tile $(t,p_n)$ of $B$ has $s^{B}_{t,p_n}$, the contribution to an output tile is

$$
C_{p_m,p_n}
=
\sum_t
\big(s^{A}_{p_m,t} A_{p_m,t}\big)
\big(s^{B}_{t,p_n} B_{t,p_n}\big),
$$

with dequant fused into the $K$-loop loads (official tutorial) instead of writing full $\mathrm{fp16}$ copies of $A,B$. Same blocked tile grid as tutorial 03.

**Running example.** $M=256$, $K=128$, $N=512$, $s_A=0.5$, $s_B=2.0$ → $C = AB$ because $s_A s_B = 1$.

### Theory board

![Whiteboard: scale-then-GEMM vs in-kernel block scales](docs/images/whiteboard-theory.jpg)

*Theory board — materialize scaled $A,B$ vs dequant with per-block scales inside the $K$-loop.*

### Pitfalls

- Scalar teaching scales ≠ per-block FP8 microscales.
- Materializing full dequant tensors wastes HBM vs in-loop scale.
- Accumulate in higher precision after dequant.

### Self-check

1. Prove C=(sA A)(sB B)=(sA sB)AB for scalars.
2. Where would per-block scales be read in the K-loop?
3. Why FP8 needs scales at all?


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import importlib.util

import pytest
import torch
import triton
import triton.language as tl

def _active_device():
    """Pick Triton active device, else CUDA, else CPU (study helper; no common/ package)."""
    try:
        import triton

        try:
            return triton.runtime.driver.active.get_active_torch_device()
        except Exception:
            pass
    except Exception:
        pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = _active_device()


def require_gpu() -> None:
    if not torch.cuda.is_available() and getattr(DEVICE, "type", None) != "cuda":
        pytest.skip(f"GPU not available (active device: {DEVICE})")


_matmul_path = Path(__file__).resolve().parents[1] / "03_matrix_multiplication" / "tutorial.py"
_spec = importlib.util.spec_from_file_location("matmul_mod", _matmul_path)
_matmul_mod = importlib.util.module_from_spec(_spec)
assert _spec.loader is not None
_spec.loader.exec_module(_matmul_mod)
matmul = _matmul_mod.matmul


## 1–3. Ladder: Naive → Intermediate → Triton sketch → FP8 blocks

| Stage | Realization | Memory story |
|---|---|---|
| **Naive** | Materialize $\tilde A=s_A A$, $\tilde B=s_B B$, then $\tilde A\tilde B$ | Two full scaled tensors + GEMM |
| **Intermediate** | Same scaling in PyTorch; GEMM via tutorial-03 `matmul` | Same math; Triton GEMM body |
| **Triton sketch** | Elementwise `y=x*s` kernel, then grouped matmul | Separable teaching steps |
| **Official** | In-kernel per-block / FP8 scales next to each $K$-panel | No full dequant copies |

**Equivalence (scalars).** All three notebook paths compute $(s_A s_B)AB$ in exact arithmetic; FP differences come only from rounding order.

**Why block scales?** Per-tensor scales collapse dynamic range under outliers; per-block (MX-style) scales adapt locally for little metadata — the usual path into FP8 LLM GEMMs.

**Next step.** Replace scalars with a grid of $s^{A}_{p_m,t}$, $s^{B}_{t,p_n}$ and measure $\\|AB-A_q B_q\\|_F/\\|AB\\|_F$; then read the official fused FP8 kernel.


In [ ]:
def naive_block_scaled_matmul_pseudo(a, b, scale_a, scale_b):
    """
    Materialize scaled A/B in memory, then matmul.

    Memory: two full scaled tensors + the GEMM.
    Fine for scalars; wasteful when scales are per-block and you could
    dequantize inside the GEMM tile loop (official FP8 path).
    """
    return torch.matmul(a * scale_a, b * scale_b)


In [ ]:
def block_scaled_matmul_ref(a, b, scale_a, scale_b):
    """
    Intermediate: still scale in PyTorch, but GEMM via Triton tutorial 03.
    """
    return matmul(a * scale_a, b * scale_b)


In [ ]:
@triton.jit
def scale_kernel(x_ptr, y_ptr, scale, n_elements, BLOCK_SIZE: tl.constexpr):
    """Elementwise y = x * scale — stand-in for on-the-fly dequant loads."""
    pid = tl.program_id(0)
    offs = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask)
    tl.store(y_ptr + offs, x * scale, mask=mask)


def triton_scale(x, scale):
    y = torch.empty_like(x)
    n = x.numel()
    grid = lambda meta: (triton.cdiv(n, meta["BLOCK_SIZE"]),)
    scale_kernel[grid](x, y, float(scale), n, BLOCK_SIZE=1024)
    return y


def block_scaled_matmul_triton(a, b, scale_a, scale_b):
    """
    Triton path: scale with a tiny kernel, then grouped matmul from tutorial 03.

    Real block-scaled FP8 kernels fuse the scale lookup into the K-loop loads;
    this keeps the teaching steps separable.
    """
    return matmul(triton_scale(a, scale_a), triton_scale(b, scale_b))


In [ ]:
def test_pseudo_matches_torch() -> None:
    torch.manual_seed(0)
    a = torch.randn(32, 16)
    b = torch.randn(16, 24)
    sa, sb = 0.5, 2.0
    ref = (a * sa) @ (b * sb)
    assert torch.allclose(naive_block_scaled_matmul_pseudo(a, b, sa, sb), ref, atol=1e-5)


def test_correctness() -> None:
    require_gpu()
    m, k, n = 256, 128, 512
    a = torch.randn(m, k, device=DEVICE, dtype=torch.float16)
    b = torch.randn(k, n, device=DEVICE, dtype=torch.float16)
    sa = torch.tensor(0.5, device=DEVICE, dtype=torch.float16)
    sb = torch.tensor(2.0, device=DEVICE, dtype=torch.float16)
    ref = torch.matmul(a * sa, b * sb)
    c_ref = block_scaled_matmul_ref(a, b, sa, sb)
    c_tl = block_scaled_matmul_triton(a, b, float(sa), float(sb))
    assert torch.allclose(c_ref.float(), ref.float(), atol=1e-2, rtol=1e-2)
    assert torch.allclose(c_tl.float(), ref.float(), atol=1e-2, rtol=1e-2)
    print("block-scaled OK")
